In [ ]:

import pandas as pd

# Define the path for the training and test datasets
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/cirrhosis_patient/train.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/cirrhosis_patient/test.csv'

# Load the datasets
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Display the first few rows of both datasets
train_df.head(), test_df.head()


(     id  N_Days             Drug    Age  ... Platelets Prothrombin Stage Status
 0  6703    1153          Placebo  14772  ...     236.0         9.9   3.0     CL
 1  5815    1447          Placebo  14754  ...     306.0         9.5   2.0      C
 2  3429    2891          Placebo  14899  ...     322.0         9.5   2.0      C
 3  2405     334  D-penicillamine  22369  ...     156.0        11.0   2.0      C
 4  1410    3820          Placebo  20597  ...     119.0        11.7   4.0      D
 
 [5 rows x 20 columns],
      id  N_Days             Drug    Age  ... Platelets Prothrombin Stage Status
 0  3467     971  D-penicillamine  20555  ...     265.0         9.8   3.0      C
 1   465    1810  D-penicillamine  20555  ...     341.0        10.6   4.0      C
 2   453    2176          Placebo  17263  ...     223.0         9.9   3.0      C
 3  4913    1462          Placebo  23331  ...     284.0        10.5   3.0      D
 4  5907    1216          Placebo  19994  ...     256.0         9.5   1.0      C
 


In [ ]:

# Step 1: Display basic information about the datasets
train_df.info()
test_df.info()

# Step 2: Display descriptive statistics for the numerical columns
train_df.describe()

# Step 3: Check for missing values in the datasets
train_df.isnull().sum(), test_df.isnull().sum()


Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

on-null   object 
 3   Age            6324 non-null   int64  
 4   Sex            6324 non-null   object 
 5   Ascites        6324 non-null   object 
 6   Hepatomegaly   6324 non-null   object 
 7   Spiders        6324 non-null   object 
 8   Edema          6324 non-null   object 
 9   Bilirubin      6324 non-null   float64
 10  Cholesterol    6324 non-null   float64
 11  Albumin        6324 non-null   float64
 12  Copper         6324 non-null   float64
 13  Alk_Phos       6324 non-null   float64
 14  SGOT           6324 non-null   float64
 15  Tryglicerides  6324 non-null   float64
 16  Platelets      6324 non-null   float64
 17  Prothrombin    6324 non-null   float64
 18  Stage          6324 non-null   float64
 19  Status         6324 non-null   object 
dtypes: float64(10), int64(3), object(7)
memory usage: 988.2+ KB
<cl

In [ ]:

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Identify categorical and numerical columns
categorical_cols = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Status']
numerical_cols = [col for col in train_df.columns if col not in categorical_cols and col != 'id' and col != 'N_Days']

# Initialize encoders and scaler
label_encoders = {}
scaler = StandardScaler()

# Encode categorical columns
for col in categorical_cols:
    label_encoders[col] = LabelEncoder()
    train_df[col] = label_encoders[col].fit_transform(train_df[col])
    test_df[col] = label_encoders[col].transform(test_df[col])

# Scale numerical columns
train_df[numerical_cols] = scaler.fit_transform(train_df[numerical_cols])
test_df[numerical_cols] = scaler.transform(test_df[numerical_cols])

# Display the first few rows of the preprocessed datasets
train_df.head(), test_df.head()


(     id  N_Days  Drug       Age  ...  Platelets  Prothrombin     Stage  Status
 0  6703    1153     1 -0.969151  ...  -0.335471    -0.925848 -0.040351       1
 1  5815    1447     1 -0.974053  ...   0.465361    -1.434841 -1.195018       0
 2  3429    2891     1 -0.934562  ...   0.648408    -1.434841 -1.195018       0
 3  2405     334     0  1.099898  ...  -1.250708     0.473880 -1.195018       0
 4  1410    3820     1  0.617292  ...  -1.674005     1.364617  1.114316       2
 
 [5 rows x 20 columns],
      id  N_Days  Drug       Age  ...  Platelets  Prothrombin     Stage  Status
 0  3467     971     0  0.605853  ...  -0.003698    -1.053097 -0.040351       0
 1   465    1810     0  0.605853  ...   0.865777    -0.035112  1.114316       0
 2   453    2176     1 -0.290725  ...  -0.484197    -0.925848 -0.040351       0
 3  4913    1462     1  1.361899  ...   0.213671    -0.162360 -0.040351       2
 4  5907    1216     1  0.453065  ...  -0.106662    -1.434841 -2.349685       0
 
 [5 rows x 2

In [ ]:


# Split the training data into features and target
X_train = train_df.drop(['id', 'N_Days'], axis=1)
y_train = train_df['N_Days']

# Split the test data into features and target
X_test = test_df.drop(['id', 'N_Days'], axis=1)
y_test = test_df['N_Days']

# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Display the shapes of the datasets
X_train.shape, X_val.shape, y_train.shape, y_val.shape, X_test.shape, y_test.shape



((5059, 18), (1265, 18), (5059,), (1265,), (1581, 18), (1581,))

In [ ]:


import xgboost as xgb
from sklearn.metrics import mean_squared_error, log_loss

# Initialize the XGBoost regressor
model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

# Train the model on the training data
model.fit(X_train, y_train)

# Make predictions on the validation set
y_val_pred = model.predict(X_val)

# Calculate the mean squared error and log loss on the validation set
mse_val = mean_squared_error(y_val, y_val_pred)
logloss_val = log_loss(y_val, y_val_pred)

mse_val, logloss_val



Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\metrics\_classification.py:3013, in log_loss(y_true, y_pred, normalize, sample_weight, labels)
   3011 if len(lb.classes_) != y_pred.shape[1]:
   3012     if labels is None:
-> 3013         raise ValueError(
   3014             "y_true and y_pred contain different number of "
   3015             "classes {0}, {1}. Please provide the true "
   3016             "labels explicitly through the labels argument. "
   3017             "Classes found in "
   3018             "y_true: {2}".format(
   3019                 transformed_labels.shape[1], y_pred.shape[1], lb.classes_
   3020             )
   3021         )
   3022     else:
   3023         raise ValueError(
   3024             "The number of classes in labels is different "
  

In [ ]:



# Make predictions on the test set
y_test_pred = model.predict(X_test)

# Calculate the mean squared error on the test set
mse_test = mean_squared_error(y_test, y_test_pred)

# Output the mean squared error
mse_test


914897.0625

In [ ]:



from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Calculate the root mean squared error (RMSE) on the test set
rmse_test = np.sqrt(mse_test)

# Calculate the R-squared score on the test set
r2_test = r2_score(y_test, y_test_pred)

# Output the RMSE and R-squared score
mse_test, rmse_test, r2_test



(914897.0625, np.float64(956.5025156788664), 0.26747775077819824)